# Modern Transport Profiles

This notebook shows the current transport postprocessing workflow built on the modern API.

It covers:
- flux-surface reduced Bohm, gyro-Bohm, and mixed transport profiles
- collisionality-dependent pinch profile built from the same reduced quantities
- projection of 1D surface-derived profiles back onto the 2D solution for plotting

At this stage the transport model is the surface-based reference implementation. A later comparison notebook can add the fully local 2D diagnostic version on top of the same plots.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from hdg_postprocess.api import (
    configure_solution_setup,
    load_reference_element,
    load_solution,
)


## Load the demo solution

This uses the same `power_balance_with_cooling` case as the focused transport tests.


In [ ]:
repo_root = Path.cwd().resolve()
while not (repo_root / "hdg_postprocess").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

demo_data = repo_root / "demos" / "data"

solution = load_solution(
    str(demo_data / "solutions" / "power_balance_boundary_checks") + "/",
    "solution_west_heating_cooling_factor",
    n_partitions=1,
)

solution.mesh.metadata.reference_element = load_reference_element(
    str(demo_data / "reference_elements" / "reference_triangle_P8.mat")
)

configure_solution_setup(
    solution,
    reference_element=str(demo_data / "reference_elements" / "reference_triangle_P8.mat"),
    radiation_model="nitrogen_cooling",
    atomic_data_dir=str(demo_data / "atomic"),
    neutral_diffusion=True,
)

solution.parameters["physics"]["R_E"] = 1.0


## Choose a flux-surface grid and transport settings

The default derivative mode now uses HDG gradients projected onto the flux-normal direction.


In [ ]:
rho_full_max = float(np.nanmax(solution.flux_surface.rho(target="node")))
rho = np.linspace(0.0, rho_full_max, 96)
method = "gauss_shell"
width = 2e-3
rho_inner = 0.8
rho_edge = 0.99
derivative_mode = "flux_normal"


## Compute surface-based transport and pinch profiles

`delta_te` is still the same non-local edge quantity used by the canonical mixed Bohm/gyro-Bohm model.


In [ ]:
bohm = solution.transport.bohm(
    rho,
    rho_inner=rho_inner,
    rho_edge=rho_edge,
    method=method,
    width=width,
    derivative_mode=derivative_mode,
)

gyrobohm = solution.transport.gyrobohm(
    rho,
    rho_inner=rho_inner,
    rho_edge=rho_edge,
    method=method,
    width=width,
    derivative_mode=derivative_mode,
)

mixed = solution.transport.bohm_gyrobohm(
    rho,
    rho_inner=rho_inner,
    rho_edge=rho_edge,
    method=method,
    width=width,
    derivative_mode=derivative_mode,
)

pinch = solution.flux_surface.pinch_velocity(
    rho,
    mixed["diffusion"],
    rho_edge=rho_edge,
    method=method,
    width=width,
)


In [ ]:
collisionality = solution.flux_surface.collisionality(rho, method=method, width=width)
pinch_factor = solution.flux_surface.pinch_factor(rho, method=method, width=width)
summary = {
    "delta_te": mixed["delta_te"],
    "rho_max": rho_full_max,
    "chi_bohm_range": (float(np.nanmin(bohm["chi_bohm"])), float(np.nanmax(bohm["chi_bohm"]))),
    "chi_gyrobohm_range": (float(np.nanmin(gyrobohm["chi_gyrobohm"])), float(np.nanmax(gyrobohm["chi_gyrobohm"]))),
    "chi_i_range": (float(np.nanmin(mixed["chi_i"])), float(np.nanmax(mixed["chi_i"]))),
    "chi_e_range": (float(np.nanmin(mixed["chi_e"])), float(np.nanmax(mixed["chi_e"]))),
    "diffusion_range": (float(np.nanmin(mixed["diffusion"])), float(np.nanmax(mixed["diffusion"]))),
    "collisionality_range": (float(np.nanmin(collisionality)), float(np.nanmax(collisionality))),
    "pinch_factor_range": (float(np.nanmin(pinch_factor)), float(np.nanmax(pinch_factor))),
    "pinch_range": (float(np.nanmin(pinch)), float(np.nanmax(pinch))),
}
summary


## Plot the 1D profiles

This is the first comparison surface to inspect before introducing the local 2D diagnostic transport path.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

axes[0, 0].plot(rho, bohm["chi_bohm"], label="Bohm")
axes[0, 0].plot(rho, gyrobohm["chi_gyrobohm"], label="Gyro-Bohm")
axes[0, 0].set_yscale("log")
axes[0, 0].set_xlabel(r"$\rho_{pol,norm}$")
axes[0, 0].set_ylabel(r"m$^2$/s")
axes[0, 0].set_title("Model components")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(rho, mixed["chi_i"], label=r"$\chi_i$ ion heat diffusivity")
axes[0, 1].plot(rho, mixed["chi_e"], label=r"$\chi_e$ electron heat diffusivity")
axes[0, 1].plot(rho, mixed["diffusion"], label="D particle diffusion")
axes[0, 1].set_xlabel(r"$\rho_{pol,norm}$")
axes[0, 1].set_ylabel(r"m$^2$/s")
axes[0, 1].set_title("Transport outputs")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(rho, collisionality, label=r"$\nu_e^*$")
axes[1, 0].set_yscale("log")
axes[1, 0].set_xlabel(r"$\rho_{pol,norm}$")
axes[1, 0].set_title("Collisionality")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(rho, pinch_factor, label="pinch factor")
axes[1, 1].plot(rho, pinch, label="pinch velocity")
axes[1, 1].set_yscale("log")
axes[1, 1].set_xlabel(r"$\rho_{pol,norm}$")
axes[1, 1].set_title("Pinch response")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.show()


## Project the 1D surface-derived profiles back to 2D

This keeps the transport model surface-based, but already lets you inspect the corresponding 2D fields on the mesh.


In [ ]:
chi_i_node = solution.flux_surface.project(rho, mixed["chi_i"], target="node")
chi_e_node = solution.flux_surface.project(rho, mixed["chi_e"], target="node")
d_node = solution.flux_surface.project(rho, mixed["diffusion"], target="node")
pinch_node = solution.flux_surface.project(rho, pinch, target="node")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

solution.mesh.plot.full(
    chi_i_node,
    ax=axes[0, 0],
    connectivity=solution.mesh.geometry.connectivity_big,
    n_levels=80,
    label=r"$\chi_i$ [m$^2$/s]",
)
axes[0, 0].set_title("Projected ion diffusivity")

solution.mesh.plot.full(
    chi_e_node,
    ax=axes[0, 1],
    connectivity=solution.mesh.geometry.connectivity_big,
    n_levels=80,
    label=r"$\chi_e$ [m$^2$/s]",
)
axes[0, 1].set_title("Projected electron diffusivity")

solution.mesh.plot.full(
    d_node,
    ax=axes[1, 0],
    connectivity=solution.mesh.geometry.connectivity_big,
    n_levels=80,
    label=r"$D$ [m$^2$/s]",
)
axes[1, 0].set_title("Projected particle diffusion")

solution.mesh.plot.full(
    pinch_node,
    ax=axes[1, 1],
    connectivity=solution.mesh.geometry.connectivity_big,
    n_levels=80,
    label=r"$v_{pinch}$ [m/s]",
)
axes[1, 1].set_title("Projected pinch velocity")

plt.show()


## Inspect the gradient assumptions behind the current transport model

These are the surface-averaged normalized gradients built from HDG physical gradients projected along the local flux-normal direction.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5), constrained_layout=True)
ax.plot(rho, mixed["a_over_te_dte"], label=r"$-a \, \nabla T_e \cdot \hat{n}_\psi / T_e$")
ax.plot(rho, mixed["a_over_pe_dpe"], label=r"$-a \, \nabla p_e \cdot \hat{n}_\psi / p_e$")
ax.set_xlabel(r"$\rho_{pol,norm}$")
ax.set_title(f"Gradient inputs ({mixed['derivative_mode']})")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


## Next comparison step

The missing comparison branch is the fully local 2D diagnostic transport path, where the same `delta_te` is kept but the other ingredients are evaluated locally before plotting. Once that exists, this notebook can be extended to show:
- surface-based projected 2D fields
- fully local 2D fields
- ratio and difference maps between the two assumptions
